# CNN XP Coefficient Truncation Sweep

This notebook evaluates a lightweight one-dimensional convolutional neural network across truncation levels of the Gaia XP coefficient representation. For each value of K, the network uses the first K BP coefficients and the first K RP coefficients as two aligned input channels, then estimates performance across repeated stratified train/validation/test splits.

The analysis records repeat-level metrics, summarizes each metric by K, and visualizes the main metric trends with a common figure design: mean performance, 95% confidence interval for the mean, and a LOWESS fit over repeat-level results. CNN-specific resampling surface diagnostics are included for examining the distribution of repeated-run metrics across K.


## Configuration and experiment state


In [ ]:
import os, json, time
from pathlib import Path

import numpy as np

import tensorflow as tf
from tensorflow import keras
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, confusion_matrix

# Resolve data and output paths from the project root instead of the kernel launch directory.
def find_project_root(start=None):
    marker_files = [
        "out_data/gaia_dr3_xp_c110_l2_binary.csv",
        "preparations/VOSA_labels_training.csv",
        "README.md",
    ]
    configured_root = os.environ.get("ASTROPHYSICS_PROJECT_ROOT")
    search_roots = []
    if configured_root:
        search_roots.append(Path(configured_root).expanduser())

    start = Path.cwd() if start is None else Path(start)
    for root in [start, *start.parents]:
        packaged_root = root / "prep"
        if packaged_root.exists():
            search_roots.append(packaged_root)
        search_roots.append(root)

    desktop_project = Path.home() / "Desktop" / "Astrophysics"
    if (desktop_project / "prep").exists():
        search_roots.append(desktop_project / "prep")
    if desktop_project.exists():
        search_roots.append(desktop_project)

    seen = set()
    for root in search_roots:
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)
        if any((root / marker).exists() for marker in marker_files):
            return root

    raise FileNotFoundError(
        "Could not locate the project root. Start Jupyter from the prep directory "
        "or set ASTROPHYSICS_PROJECT_ROOT to the directory that contains this truncation bundle."
    )

PROJECT_ROOT = find_project_root()
CSV_PATH = PROJECT_ROOT / "out_data" / "gaia_dr3_xp_c110_l2_binary.csv"

OUT_DIR = PROJECT_ROOT / "k55_experiments" / "cnn_k55_truncation_out"
RUNS_DIR = OUT_DIR / "runs"
MODELS_DIR = OUT_DIR / "models"
for d in [OUT_DIR, RUNS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HEARTBEAT_PATH = OUT_DIR / "HEARTBEAT.json"
LAST_EVENT_PATH = OUT_DIR / "LAST_EVENT.txt"
MANIFEST_PATH = OUT_DIR / "RUNNING_MANIFEST.json"

def hb(payload):
    payload = dict(payload); payload["time"] = time.ctime()
    HEARTBEAT_PATH.write_text(json.dumps(payload, indent=2))

def log_event(msg):
    LAST_EVENT_PATH.write_text(f"[{time.ctime()}] {msg}\n")

def _safe_html(s):
    return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

class NotebookProgress:
    def __init__(self, total, desc="Progress"):
        self.total = max(int(total), 1)
        self.desc = desc
        self.n = 0
        self.postfix = ""
        self._mode = "text"
        self._label = None
        self._bar = None
        self._text_render(force=True)
        try:
            import ipywidgets as widgets
            from IPython.display import display
            self._label = widgets.HTML(value=self._html())
            self._bar = widgets.IntProgress(value=0, min=0, max=self.total)
            display(widgets.VBox([self._label, self._bar]))
            self._mode = "widget"
        except Exception:
            pass

    def _html(self):
        base = f"<b>{_safe_html(self.desc)}</b>: {self.n}/{self.total}"
        if self.postfix:
            base += f" | {_safe_html(self.postfix)}"
        return base

    def _text_render(self, force=False):
        if force:
            line = f"{self.desc}: {self.n}/{self.total}"
            if self.postfix:
                line += f" | {self.postfix}"
            print(line, flush=True)

    def set_postfix(self, text):
        self.postfix = str(text)
        if self._mode == "widget":
            self._label.value = self._html()

    def update(self, n=1):
        self.n = min(self.total, self.n + int(n))
        if self._mode == "widget":
            self._bar.value = self.n
            self._label.value = self._html()
        else:
            self._text_render(force=True)

    def close(self):
        if self._mode == "widget" and self._bar is not None:
            self._bar.bar_style = "success"
            self._label.value = self._html()
        else:
            self._text_render(force=True)

# ------------------------
# Experiment config
# ------------------------
K_MIN, K_MAX = 1, 55
K_LIST = list(range(K_MIN, K_MAX+1))

N_REPEATS = 100
TEST_FRAC = 0.15
VAL_FRAC = 0.15
VAL_FRAC_OF_REMAIN = VAL_FRAC / (1.0 - TEST_FRAC)

BASE_SEED = 1234
THR_GRID_SIZE = 800

EPOCHS = 260
PATIENCE = 35
BATCH_SIZE = 32
LR = 1e-3

tf.config.optimizer.set_jit(True)
try:
    tf.config.threading.set_inter_op_parallelism_threads(2)
    tf.config.threading.set_intra_op_parallelism_threads(6)
except Exception:
    pass

manifest = dict(
    started_at=time.ctime(),
    csv=str(CSV_PATH),
    out_dir=str(OUT_DIR.resolve()),
    plan=dict(K_MIN=K_MIN, K_MAX=K_MAX, N_REPEATS=N_REPEATS, TEST_FRAC=TEST_FRAC, VAL_FRAC=VAL_FRAC,
              EPOCHS=EPOCHS, PATIENCE=PATIENCE, BATCH_SIZE=BATCH_SIZE, LR=LR, THR_GRID_SIZE=THR_GRID_SIZE),
    tf_version=tf.__version__,
    gpus=[str(x) for x in tf.config.list_physical_devices("GPU")]
)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
log_event("Manifest created.")
hb({"status":"ready"})

print("OUT_DIR:", OUT_DIR.resolve())
print("TF GPUs:", tf.config.list_physical_devices("GPU"))


## Load Gaia XP coefficient dataset


In [ ]:
df = pd.read_csv(CSV_PATH)
if "y" not in df.columns:
    raise KeyError("CSV must contain column 'y' for labels")

df["y"] = df["y"].astype(int)

C110 = [f"c{i:03d}" for i in range(110)]
missing = [c for c in C110 if c not in df.columns]
if missing:
    raise KeyError(f"Missing c110 columns, e.g. {missing[:6]}")

BP = [f"c{i:03d}" for i in range(55)]          # c000..c054
RP = [f"c{i:03d}" for i in range(55, 110)]     # c055..c109

Xbp = df[BP].to_numpy(np.float32)
Xrp = df[RP].to_numpy(np.float32)
y_all = df["y"].to_numpy(int)

print("Xbp:", Xbp.shape, "Xrp:", Xrp.shape, "pos_rate:", float(y_all.mean()))


## Create stratified train, validation, and test splits


In [ ]:
def make_splits(y, n_repeats, base_seed):
    splitter = StratifiedShuffleSplit(n_splits=n_repeats, test_size=TEST_FRAC, random_state=base_seed)
    idx_all = np.arange(len(y))
    splits = []
    for r, (trainval_rel, test_rel) in enumerate(splitter.split(np.zeros(len(y)), y)):
        idx_trainval = idx_all[trainval_rel]
        idx_test = idx_all[test_rel]

        y_tv = y[idx_trainval]
        splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRAC_OF_REMAIN, random_state=base_seed + 1000 + r)
        tr_rel2, va_rel2 = next(splitter2.split(np.zeros(len(y_tv)), y_tv))

        idx_train = idx_trainval[tr_rel2]
        idx_val   = idx_trainval[va_rel2]
        splits.append(dict(repeat=r, train_idx=idx_train, val_idx=idx_val, test_idx=idx_test))
    return splits

splits = make_splits(y_all, N_REPEATS, BASE_SEED)
print("splits:", len(splits), "sizes:", {k: len(splits[0][k]) for k in ["train_idx","val_idx","test_idx"]})


## Construct truncated CNN inputs


In [ ]:
def make_X_for_K(idx, K):
    """
    Keep first K BP coeffs and first K RP coeffs.
    Return X shape (N, K, 2) with channels: [BP, RP].
    """
    bp = Xbp[idx, :K]  # (N,K)
    rp = Xrp[idx, :K]  # (N,K)
    X = np.stack([bp, rp], axis=-1)  # (N,K,2)
    return X

# quick check
Xdemo = make_X_for_K(np.arange(5), 10)
print("demo shape:", Xdemo.shape)


## Metric and threshold helpers


In [ ]:
def prob_metrics(y_true, p):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    pc = np.clip(p, 1e-15, 1-1e-15)
    return dict(
        ROC_AUC=roc_auc_score(y_true, p) if len(np.unique(y_true))>1 else np.nan,
        PR_AUC=average_precision_score(y_true, p) if len(np.unique(y_true))>1 else np.nan,
        Brier=brier_score_loss(y_true, p),
        LogLoss=log_loss(y_true, pc, labels=[0,1]),
    )

def confusion_metrics(y_true, y_hat):
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
    sens = tp/(tp+fn) if (tp+fn) else np.nan
    spec = tn/(tn+fp) if (tn+fp) else np.nan
    prec = tp/(tp+fp) if (tp+fp) else np.nan
    return prec, sens, spec, tn, fp, fn, tp

def threshold_grid(p, n=800):
    lo, hi = float(np.min(p)), float(np.max(p))
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        return np.array([0.5])
    return np.linspace(lo, hi, n)

def choose_threshold(y_true, p, criterion="youden", n=800):
    best_thr, best_score = 0.5, -np.inf
    for thr in threshold_grid(p, n):
        yhat = (p >= thr).astype(int)
        prec, sens, spec, *_ = confusion_metrics(y_true, yhat)
        if criterion == "youden":
            score = (sens + spec - 1) if np.isfinite(sens) and np.isfinite(spec) else -np.inf
        elif criterion == "f1":
            score = (2*prec*sens/(prec+sens)) if (np.isfinite(prec) and np.isfinite(sens) and (prec+sens)>0) else -np.inf
        else:
            raise ValueError
        if score > best_score:
            best_score, best_thr = float(score), float(thr)
    return best_thr, float(best_score)

def evaluate(yv, pv, yt, pt):
    out = {f"test_{k}": float(v) for k,v in prob_metrics(yt, pt).items()}
    for crit in ["youden","f1"]:
        thr, sc = choose_threshold(yv, pv, crit, THR_GRID_SIZE)
        yhat = (pt >= thr).astype(int)
        prec, sens, spec, tn, fp, fn, tp = confusion_metrics(yt, yhat)
        out[f"{crit}_val_thr"] = thr
        out[f"{crit}_val_score"] = sc
        out[f"{crit}_test_Precision"] = float(prec)
        out[f"{crit}_test_Sensitivity"] = float(sens)
        out[f"{crit}_test_Specificity"] = float(spec)
        out[f"{crit}_test_tn"] = int(tn); out[f"{crit}_test_fp"] = int(fp)
        out[f"{crit}_test_fn"] = int(fn); out[f"{crit}_test_tp"] = int(tp)
    return out


## CNN model and training helpers


In [ ]:
def light_respectable_cnn(input_shape, base_ch=32, k=5, blocks=3, dropout=0.25, bn=True):
    inp = keras.layers.Input(shape=input_shape)
    x = inp
    ch = base_ch
    for _ in range(blocks):
        x = keras.layers.Conv1D(ch, k, padding="same")(x)
        if bn:
            x = keras.layers.BatchNormalization()(x)
        x = keras.layers.LeakyReLU()(x)
        x = keras.layers.MaxPool1D(pool_size=2, padding="same")(x)
        x = keras.layers.Dropout(dropout)(x)
        ch = min(ch*2, 128)

    x = keras.layers.GlobalAveragePooling1D()(x)
    x = keras.layers.Dense(64)(x)
    x = keras.layers.LeakyReLU()(x)
    x = keras.layers.Dropout(dropout)(x)
    out = keras.layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inp, out)
    opt = keras.optimizers.Adam(learning_rate=LR)
    model.compile(optimizer=opt, loss="binary_crossentropy")
    return model

class EpochHeartbeat(keras.callbacks.Callback):
    def __init__(self, tag):
        super().__init__()
        self.tag = tag
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = float(logs.get("loss", np.nan))
        val_loss = float(logs.get("val_loss", np.nan))
        hb({
            "status": "training",
            "tag": self.tag,
            "epoch": int(epoch),
            "loss": loss,
            "val_loss": val_loss,
        })
        print(f"[{self.tag}] epoch {int(epoch)+1}/{EPOCHS} loss={loss:.5f} val_loss={val_loss:.5f}", flush=True)

def train_one(Xtr, ytr, Xva, yva, model_path: Path, tag: str):
    tf.keras.backend.clear_session()
    tf.random.set_seed(BASE_SEED + (hash(tag) % 1000000))

    model = light_respectable_cnn(input_shape=Xtr.shape[1:])
    best_model_path = model_path.with_name(f"{model_path.stem}.best.keras")

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=False),
        keras.callbacks.ModelCheckpoint(str(best_model_path), monitor="val_loss",
                                        save_best_only=True, save_weights_only=False),
        EpochHeartbeat(tag),
    ]

    model.fit(Xtr, ytr, validation_data=(Xva,yva),
              epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, callbacks=callbacks)

    model = keras.models.load_model(best_model_path)
    model.save(model_path)
    return model


## Shard and model persistence helpers


In [ ]:
def run_id(K, rep):
    return f"K{int(K):02d}__r{int(rep):02d}"

def shard_path(K, rep):
    return RUNS_DIR / f"{run_id(K, rep)}.parquet"

def done(K, rep):
    return shard_path(K, rep).exists()

def save_row(row):
    pd.DataFrame([row]).to_parquet(RUNS_DIR / f"{row['run_id']}.parquet", index=False)

def load_all_rows():
    files = sorted(RUNS_DIR.glob("*.parquet"))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)

log_event("Ready to run truncation sweep.")
hb({"status":"ready"})
print("RUNS_DIR:", RUNS_DIR.resolve())


## Use precomputed results or run the truncation sweep


In [ ]:
# Execution controls for the K55 sweep.
USE_PRECOMPUTED_K55_RESULTS = bool(globals().get("USE_PRECOMPUTED_K55_RESULTS", True))
RUN_K55_SWEEP = bool(globals().get("RUN_K55_SWEEP", False))
PRECOMPUTED_RESULT_BASENAME = "truncation_cnn_raw.csv"
PRECOMPUTED_RESULT_PATHS = [
    Path("k55_experiments/cnn_k55_truncation_out/truncation_cnn_raw.parquet"),
    Path("k55_experiments/cnn_k55_truncation_out/truncation_cnn_raw.csv"),
    Path("k55_experiments/cnn_k55_truncation_out/truncation_cnn_raw.parquet"),
    Path("k55_experiments/cnn_k55_truncation_out/truncation_cnn_raw.csv"),
]


def _resolve_project_path(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def _precomputed_candidate_paths():
    candidates = [
        OUT_DIR / PRECOMPUTED_RESULT_BASENAME.replace(".csv", ".parquet"),
        OUT_DIR / PRECOMPUTED_RESULT_BASENAME,
    ]
    candidates.extend(_resolve_project_path(path) for path in PRECOMPUTED_RESULT_PATHS)

    unique = []
    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        key = str(candidate.resolve()) if candidate.exists() else str(candidate)
        if key not in seen:
            unique.append(candidate)
            seen.add(key)
    return unique


def _read_result_pairs(path):
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path, columns=["K", "repeat"])
    return pd.read_csv(path, usecols=["K", "repeat"])


def _is_complete_precomputed_result(path):
    try:
        pairs = _read_result_pairs(path).dropna().copy()
    except Exception as exc:
        print(f"Rejected precomputed result table {path}: {exc}", flush=True)
        return False

    pairs["K"] = pairs["K"].astype(int)
    pairs["repeat"] = pairs["repeat"].astype(int)
    pairs = pairs.drop_duplicates(["K", "repeat"])
    expected = len(K_LIST) * N_REPEATS
    if len(pairs) < expected:
        print(f"Rejected precomputed result table {path}: {len(pairs)}/{expected} unique K-repeat rows", flush=True)
        return False

    counts = pairs[pairs["K"].isin(K_LIST)].groupby("K")["repeat"].nunique()
    complete = len(counts) == len(K_LIST) and int(counts.min()) >= N_REPEATS
    if not complete:
        found = int(counts.sum()) if not counts.empty else 0
        print(f"Rejected precomputed result table {path}: incomplete K-repeat coverage ({found}/{expected})", flush=True)
    return complete


def find_complete_precomputed_result():
    if not USE_PRECOMPUTED_K55_RESULTS:
        return None
    for candidate in _precomputed_candidate_paths():
        if not candidate.exists():
            continue
        print("Checking precomputed result table:", candidate, flush=True)
        if _is_complete_precomputed_result(candidate):
            return candidate
    return None


precomputed_result_path = find_complete_precomputed_result()
if precomputed_result_path is not None:
    log_event(f"Using precomputed truncation results: {precomputed_result_path}")
    hb({"status": "precomputed_results_available", "path": str(precomputed_result_path)})
    print("Complete precomputed result table found:", precomputed_result_path, flush=True)
    print("Skipping the model sweep. Continue with the collection and plotting cells.", flush=True)
elif not RUN_K55_SWEEP:
    log_event("No complete precomputed K-sweep result table was found; sweep not started.")
    hb({"status": "k55_sweep_not_started"})
    print("No complete precomputed K-sweep result table was found.", flush=True)
    print("Set RUN_K55_SWEEP = True before this cell to run the K55 sweep.", flush=True)
else:
    total_runs = len(K_LIST) * N_REPEATS
    resumed_runs = sum(1 for K in K_LIST for sp in splits if done(K, sp["repeat"]))
    print(f"Resuming from saved runs: {resumed_runs}/{total_runs} ({resumed_runs/total_runs:.1%})")

    run_bar = NotebookProgress(total_runs, desc="Runs (K,repeat)")
    if resumed_runs:
        run_bar.update(resumed_runs)

    for K in K_LIST:
        log_event(f"Starting K={K}")

        for sp in splits:
            rep = sp["repeat"]
            tag = f"K={K}|r={rep}"
            run_bar.set_postfix(tag)

            if done(K, rep):
                continue

            hb({"status":"run_start", "tag": tag})
            print(f"Starting run {tag}", flush=True)

            tr, va, te = sp["train_idx"], sp["val_idx"], sp["test_idx"]

            Xtr = make_X_for_K(tr, K); ytr = y_all[tr]
            Xva = make_X_for_K(va, K); yva = y_all[va]
            Xte = make_X_for_K(te, K); yte = y_all[te]

            model_path = MODELS_DIR / f"{run_id(K, rep)}.keras"

            t0 = time.time()
            model = train_one(Xtr, ytr, Xva, yva, model_path=model_path, tag=tag)

            pv = model.predict(Xva, batch_size=1024, verbose=0).reshape(-1)
            pt = model.predict(Xte, batch_size=1024, verbose=0).reshape(-1)

            met = evaluate(yva, pv, yte, pt)
            dt = time.time() - t0

            row = dict(
                run_id=run_id(K, rep),
                K=int(K),
                repeat=int(rep),
                model_path=str(model_path),
                runtime_s=float(dt),
                n_train=int(len(ytr)), n_val=int(len(yva)), n_test=int(len(yte)),
                test_pos_rate=float(np.mean(yte)),
                **met
            )
            save_row(row)

            hb({"status":"run_done", "tag": tag, "runtime_s": dt})
            print(f"Finished run {tag} in {dt/60.0:.2f} min", flush=True)
            run_bar.update(1)

    run_bar.close()

    log_event("ALL DONE.")
    hb({"status":"completed"})
    print("Done. Shards:", len(list(RUNS_DIR.glob('*.parquet'))), "Models:", len(list(MODELS_DIR.glob('*.keras'))))



## Collect results and summarize by K


In [ ]:
REPORT_DIR = OUT_DIR / "analysis_report"
MODEL_LABEL = "CNN"
RAW_OUTPUT_BASENAME = "truncation_cnn_raw.csv"
SUMMARY_OUTPUT_BASENAME = "truncation_cnn_summary_byK.csv"
MODEL_EXTRA_METRICS = []
ALTERNATE_RESULT_PATHS = [
    Path("k55_experiments/cnn_k55_truncation_out/truncation_cnn_raw.parquet"),
    Path("k55_experiments/cnn_k55_truncation_out/truncation_cnn_raw.csv"),
]

# Re-resolve path globals here so this reporting cell works even after a kernel restart.
if "find_project_root" not in globals():
    def find_project_root(start=None):
        marker_files = [
            "out_data/gaia_dr3_xp_c110_l2_binary.csv",
            "01_build_gaia_xp_coefficient_dataset.ipynb",
            "xpcoeff_feature_build_binary.ipynb",
        ]
        configured_root = os.environ.get("ASTROPHYSICS_PROJECT_ROOT")
        search_roots = []
        if configured_root:
            search_roots.append(Path(configured_root).expanduser())

        start = Path.cwd() if start is None else Path(start)
        search_roots.extend([start, *start.parents])

        desktop_project = Path.home() / "Desktop" / "Astrophysics"
        if desktop_project.exists():
            search_roots.append(desktop_project)

        seen = set()
        for root in search_roots:
            root = root.resolve()
            if root in seen:
                continue
            seen.add(root)
            if any((root / marker).exists() for marker in marker_files):
                return root

        raise FileNotFoundError(
            "Could not locate the project root. Start Jupyter from the Astrophysics directory "
            "or set ASTROPHYSICS_PROJECT_ROOT to the directory that contains the coefficient dataset."
        )

PROJECT_ROOT = find_project_root()
OUT_DIR = PROJECT_ROOT / "k55_experiments" / "cnn_k55_truncation_out"
RUNS_DIR = OUT_DIR / "runs"

# Collect completed runs and write reusable summary tables.
REPORT_DIR = OUT_DIR / "analysis_report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = REPORT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RAW_PARQUET = OUT_DIR / RAW_OUTPUT_BASENAME.replace(".csv", ".parquet")
RAW_CSV = OUT_DIR / RAW_OUTPUT_BASENAME
FORCE_REBUILD_FROM_SHARDS = bool(globals().get("FORCE_REBUILD_FROM_SHARDS", False))
SHARD_PROGRESS_EVERY = int(globals().get("SHARD_PROGRESS_EVERY", 250))
MIN_EXPECTED_RESULT_ROWS = int(globals().get("MIN_EXPECTED_RESULT_ROWS", len(K_LIST) * N_REPEATS))
REQUIRED_RESULT_COLUMNS = {"K", "repeat", "test_PR_AUC", "test_ROC_AUC", "test_LogLoss"}


def _read_result_candidate(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)


def _valid_result_table(df: pd.DataFrame) -> bool:
    return REQUIRED_RESULT_COLUMNS.issubset(df.columns) and len(df) >= MIN_EXPECTED_RESULT_ROWS


def _resolve_project_path(path: Path) -> Path:
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def _candidate_result_paths() -> list[Path]:
    candidates = [RAW_PARQUET, RAW_CSV]
    candidates.extend(_resolve_project_path(path) for path in globals().get("ALTERNATE_RESULT_PATHS", []))
    unique = []
    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        key = str(candidate.resolve()) if candidate.exists() else str(candidate)
        if key not in seen:
            unique.append(candidate)
            seen.add(key)
    return unique


def load_results_table() -> pd.DataFrame:
    if not FORCE_REBUILD_FROM_SHARDS:
        for candidate in _candidate_result_paths():
            if not candidate.exists():
                continue
            print("Checking result table:", candidate, flush=True)
            candidate_df = _read_result_candidate(candidate)
            if _valid_result_table(candidate_df):
                print("Using result table:", candidate, flush=True)
                return candidate_df
            print(
                f"  rejected {candidate}: shape={candidate_df.shape}, "
                f"required rows>={MIN_EXPECTED_RESULT_ROWS}, required columns={sorted(REQUIRED_RESULT_COLUMNS)}",
                flush=True,
            )

    shard_files = sorted(RUNS_DIR.glob("*.parquet"))
    print(f"Rebuilding result table from {len(shard_files)} shard files in {RUNS_DIR}", flush=True)
    if not shard_files:
        return pd.DataFrame()

    parts = []
    for index, shard_path in enumerate(shard_files, start=1):
        parts.append(pd.read_parquet(shard_path))
        if index == 1 or index % SHARD_PROGRESS_EVERY == 0 or index == len(shard_files):
            print(f"  loaded {index}/{len(shard_files)} shards", flush=True)
    return pd.concat(parts, ignore_index=True)


res = load_results_table()
print("Result rows:", res.shape, flush=True)
if not res.empty:
    display(res.head(5))
else:
    print("No completed run shards or consolidated result table were found. Run the sweep cell before building reports.")

if not res.empty:
    res.to_parquet(RAW_PARQUET, index=False)
    res.to_csv(RAW_CSV, index=False)
    print("Saved raw outputs:")
    print(" -", RAW_PARQUET)
    print(" -", RAW_CSV)

BASE_METRICS = [
    "runtime_s",
    "test_PR_AUC", "test_ROC_AUC", "test_LogLoss", "test_Brier",
    "youden_test_Precision", "youden_test_Sensitivity", "youden_test_Specificity", "youden_val_thr",
    "f1_test_Precision", "f1_test_Sensitivity", "f1_test_Specificity", "f1_val_thr",
]

MODEL_EXTRA_METRICS = globals().get("MODEL_EXTRA_METRICS", [])
SUMMARY_METRICS = [metric for metric in BASE_METRICS + MODEL_EXTRA_METRICS if metric in res.columns]

from scipy import stats


def summarize_metric_by_k(df, metric):
    values = df[["K", metric]].dropna().copy()
    if values.empty:
        return pd.DataFrame(columns=["K", "metric", "mean", "std", "n", "se", "ci95_lo", "ci95_hi"])

    summary = values.groupby("K")[metric].agg(["mean", "std", "count"]).reset_index()
    summary = summary.rename(columns={"count": "n"})
    summary["std"] = summary["std"].fillna(0.0)
    summary["se"] = summary["std"] / np.sqrt(summary["n"].clip(lower=1))
    summary["tcrit"] = summary["n"].apply(lambda n: stats.t.ppf(0.975, int(n) - 1) if int(n) > 1 else 0.0)
    summary["ci95_lo"] = summary["mean"] - summary["tcrit"] * summary["se"]
    summary["ci95_hi"] = summary["mean"] + summary["tcrit"] * summary["se"]
    summary.insert(1, "metric", metric)
    return summary[["K", "metric", "mean", "std", "n", "se", "ci95_lo", "ci95_hi"]]


def build_summary_tables(df, metrics):
    long_parts = [summarize_metric_by_k(df, metric) for metric in metrics]
    long_summary = pd.concat(long_parts, ignore_index=True) if long_parts else pd.DataFrame()
    if long_summary.empty:
        return long_summary, pd.DataFrame()

    wide_summary = None
    for metric in metrics:
        metric_summary = summarize_metric_by_k(df, metric).drop(columns=["metric"])
        metric_summary = metric_summary.rename(
            columns={column: f"{metric}_{column}" for column in metric_summary.columns if column != "K"}
        )
        wide_summary = metric_summary if wide_summary is None else wide_summary.merge(metric_summary, on="K", how="outer")
    wide_summary = wide_summary.sort_values("K").reset_index(drop=True)
    return long_summary, wide_summary


summary_long, summary_by_k = build_summary_tables(res, SUMMARY_METRICS) if not res.empty else (pd.DataFrame(), pd.DataFrame())

if summary_by_k.empty:
    print("No summary table written because no metric results are available.")
else:
    SUMMARY_WIDE_CSV = OUT_DIR / SUMMARY_OUTPUT_BASENAME
    SUMMARY_LONG_CSV = OUT_DIR / SUMMARY_OUTPUT_BASENAME.replace(".csv", "_long.csv")
    summary_by_k.to_csv(SUMMARY_WIDE_CSV, index=False)
    summary_long.to_csv(SUMMARY_LONG_CSV, index=False)
    print("Saved summary outputs:")
    print(" -", SUMMARY_WIDE_CSV)
    print(" -", SUMMARY_LONG_CSV)
    display(summary_by_k.head(10))


## CNN-only Gaussian and resampling surface diagnostics


In [ ]:
import plotly.graph_objects as go

def resampling_saddle_surface(metric="test_PR_AUC", n_sigma=4.0, n_x=160):
    if res.empty:
        print(f"Skipping {metric}: no rows in res")
        return
    if metric not in res.columns:
        print(f"Skipping {metric}: metric not found")
        return

    stats = (
        res[["K", metric]]
        .dropna()
        .groupby("K", as_index=False)
        .agg(mu=(metric, "mean"), sigma=(metric, "std"), n=(metric, "size"))
    )
    stats = stats[(stats["n"] >= 2) & np.isfinite(stats["sigma"]) & (stats["sigma"] > 0)].copy()
    if stats.empty:
        print(f"Skipping {metric}: not enough repeats with non-zero std")
        return

    x_min = float((stats["mu"] - n_sigma * stats["sigma"]).min())
    x_max = float((stats["mu"] + n_sigma * stats["sigma"]).max())
    x_grid = np.linspace(x_min, x_max, n_x)

    ks = stats["K"].to_numpy(float)
    mus = stats["mu"].to_numpy(float)
    sigs = stats["sigma"].to_numpy(float)

    X = np.repeat(ks[:, None], n_x, axis=1)
    Y = np.tile(x_grid, (len(ks), 1))
    Z = np.zeros_like(Y)

    for i, (mu, sigma) in enumerate(zip(mus, sigs)):
        t = (x_grid - mu) / sigma
        Z[i, :] = np.exp(-0.5 * t * t) / (sigma * np.sqrt(2.0 * np.pi))

    fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, colorscale="Viridis")])
    fig.update_layout(
        title=f"Resampling saddle surface for {metric} (Gaussian from mean/std per K)",
        scene=dict(
            xaxis_title="K (coeffs per BP/RP)",
            yaxis_title=metric,
            zaxis_title="density (from repeats)",
            aspectmode="manual",
            aspectratio=dict(x=1.8, y=1.0, z=0.8),
        ),
        scene_camera=dict(eye=dict(x=2.2, y=1.3, z=0.9)),
        height=700,
        margin=dict(l=20, r=20, t=60, b=20),
    )
    fig.show()

resampling_saddle_surface("test_PR_AUC")
resampling_saddle_surface("test_LogLoss")
resampling_saddle_surface("youden_test_Sensitivity")
resampling_saddle_surface("f1_test_Precision")


In [ ]:
import plotly.graph_objects as go

def resampling_saddle_surface_binned(metric="test_PR_AUC", n_bins=80, min_repeats=1):
    if res.empty:
        print(f"Skipping {metric}: no rows in res")
        return
    if metric not in res.columns:
        print(f"Skipping {metric}: metric not found")
        return

    df = res[["K", metric]].dropna().copy()
    if df.empty:
        print(f"Skipping {metric}: no finite values after dropna")
        return

    counts = df.groupby("K", as_index=False).size().rename(columns={"size": "n"})
    valid_k = counts.loc[counts["n"] >= min_repeats, "K"]
    df = df[df["K"].isin(valid_k)].copy()
    if df.empty:
        print(f"Skipping {metric}: no K groups with at least {min_repeats} repeats")
        return

    x_min = float(df[metric].min())
    x_max = float(df[metric].max())
    if not np.isfinite(x_min) or not np.isfinite(x_max):
        print(f"Skipping {metric}: non-finite metric range")
        return
    if x_max == x_min:
        eps = max(abs(x_min), 1.0) * 1e-6
        x_min -= eps
        x_max += eps

    bin_edges = np.linspace(x_min, x_max, n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    ks = np.sort(df["K"].unique().astype(float))
    X = np.repeat(ks[:, None], n_bins, axis=1)
    Y = np.tile(bin_centers, (len(ks), 1))
    Z = np.zeros((len(ks), n_bins), dtype=float)

    for i, k in enumerate(ks):
        vals = df.loc[df["K"] == k, metric].to_numpy(float)
        hist, _ = np.histogram(vals, bins=bin_edges, density=True)
        Z[i, :] = hist

    fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, colorscale="Viridis")])
    fig.update_layout(
        title=f"Resampling saddle surface for {metric} (raw histogram bins per K)",
        scene=dict(
            xaxis_title="K (coeffs per BP/RP)",
            yaxis_title=metric,
            zaxis_title="histogram density (raw bins)",
            aspectmode="manual",
            aspectratio=dict(x=1.8, y=1.0, z=0.8),
        ),
        scene_camera=dict(eye=dict(x=2.2, y=1.3, z=0.9)),
        height=700,
        margin=dict(l=20, r=20, t=60, b=20),
    )
    fig.show()

resampling_saddle_surface_binned("test_PR_AUC")
resampling_saddle_surface_binned("test_LogLoss")
#resampling_saddle_surface_binned("youden_test_Sensitivity")
#resampling_saddle_surface_binned("f1_test_Precision")


## Standardized metric-vs-K plots


In [ ]:
# Standardized metric-vs-K plots: mean, 95% CI, and LOWESS fit.
import re
from typing import Optional

import plotly.graph_objects as go
from statsmodels.nonparametric.smoothers_lowess import lowess

METRICS_TO_PLOT = [
    "test_PR_AUC",
    "test_ROC_AUC",
    "test_LogLoss",
    "test_Brier",
    "youden_test_Sensitivity",
    "youden_test_Specificity",
    "youden_test_Precision",
    "f1_test_Sensitivity",
    "f1_test_Specificity",
    "f1_test_Precision",
]

METRIC_LABELS = {
    "test_PR_AUC": "PR-AUC",
    "test_ROC_AUC": "ROC-AUC",
    "test_LogLoss": "Log loss",
    "test_Brier": "Brier score",
    "youden_test_Sensitivity": "Youden sensitivity",
    "youden_test_Specificity": "Youden specificity",
    "youden_test_Precision": "Youden precision",
    "f1_test_Sensitivity": "F1-threshold sensitivity",
    "f1_test_Specificity": "F1-threshold specificity",
    "f1_test_Precision": "F1-threshold precision",
}

BOUNDED_METRICS = {
    "test_PR_AUC",
    "test_ROC_AUC",
    "test_Brier",
    "youden_test_Sensitivity",
    "youden_test_Specificity",
    "youden_test_Precision",
    "f1_test_Sensitivity",
    "f1_test_Specificity",
    "f1_test_Precision",
}

PLOT_COLORS = {
    "ci": "rgba(74, 111, 165, 0.18)",
    "mean_line": "#6F7F95",
    "mean_marker": "#596A80",
    "lowess": "#0E7490",
    "grid": "rgba(154, 160, 172, 0.22)",
    "axis": "#8C95A3",
    "text": "#2F3441",
}


def lowess_fit_raw(df, metric, frac=0.22, n_points=400):
    values = df[["K", metric]].dropna().copy().sort_values("K")
    if values.empty:
        return pd.DataFrame(columns=["K", "smooth"])

    if values["K"].nunique() < 2 or len(values) < 3:
        fallback = values.groupby("K", as_index=False)[metric].mean().rename(columns={metric: "smooth"})
        return fallback

    x = values["K"].to_numpy(dtype=float)
    y = values[metric].to_numpy(dtype=float)
    frac = min(max(float(frac), 2.0 / len(values)), 1.0)
    smoothed = lowess(y, x, frac=frac, it=0, return_sorted=True)
    smoothed = pd.DataFrame(smoothed, columns=["K", "smooth"]).groupby("K", as_index=False)["smooth"].mean()

    k_dense = np.linspace(values["K"].min(), values["K"].max(), n_points)
    smooth_dense = np.interp(k_dense, smoothed["K"], smoothed["smooth"])
    return pd.DataFrame({"K": k_dense, "smooth": smooth_dense})


def safe_filename(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


NONNEGATIVE_METRICS = {
    "test_LogLoss",
    "test_Brier",
    "runtime_s",
}


def adaptive_yaxis_range(summary: pd.DataFrame, fit: pd.DataFrame, metric: str, pad_fraction: float = 0.14):
    """Return a compact y-axis range around the plotted CI, mean, and LOWESS values."""
    series = [summary["ci95_lo"], summary["ci95_hi"], summary["mean"]]
    if fit is not None and not fit.empty and "smooth" in fit.columns:
        series.append(fit["smooth"])

    values = np.concatenate([np.asarray(item, dtype=float).reshape(-1) for item in series])
    values = values[np.isfinite(values)]
    if values.size == 0:
        return None

    y_min = float(values.min())
    y_max = float(values.max())
    span = y_max - y_min
    if span <= 0:
        center = 0.5 * (y_min + y_max)
        span = max(abs(center) * 0.04, 0.01)
        y_min = center - 0.5 * span
        y_max = center + 0.5 * span
    else:
        padding = max(span * pad_fraction, 1e-9)
        y_min -= padding
        y_max += padding

    if metric in BOUNDED_METRICS:
        y_min = max(0.0, y_min)
        y_max = min(1.0, y_max)
    elif metric in NONNEGATIVE_METRICS:
        y_min = max(0.0, y_min)

    if y_max <= y_min:
        center = 0.5 * (y_min + y_max)
        y_min = center - 0.005
        y_max = center + 0.005

    return [y_min, y_max]


def plot_metric_vs_k(df, metric, frac=0.22, benchmark=None, save_png=True):
    if df.empty:
        print(f"Skipping {metric}: no result rows available.")
        return None, None, None
    if metric not in df.columns:
        print(f"Skipping {metric}: column is not present in results.")
        return None, None, None

    summary = summarize_metric_by_k(df, metric)
    if summary.empty:
        print(f"Skipping {metric}: no finite values available.")
        return None, None, None

    fit = lowess_fit_raw(df, metric, frac=frac, n_points=400)
    label = METRIC_LABELS.get(metric, metric)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=summary["K"],
        y=summary["ci95_hi"],
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=summary["K"],
        y=summary["ci95_lo"],
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor=PLOT_COLORS["ci"],
        name="95% CI for mean",
        hovertemplate="K=%{x}<br>CI lower=%{y:.4f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=summary["K"],
        y=summary["mean"],
        mode="lines+markers",
        name="mean",
        line=dict(color=PLOT_COLORS["mean_line"], width=2.2),
        marker=dict(size=5, color=PLOT_COLORS["mean_marker"], opacity=0.92),
        customdata=np.stack([summary["n"], summary["std"]], axis=-1),
        hovertemplate="K=%{x}<br>mean=%{y:.4f}<br>n=%{customdata[0]}<br>std=%{customdata[1]:.4f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=fit["K"],
        y=fit["smooth"],
        mode="lines",
        name="LOWESS fit",
        line=dict(color=PLOT_COLORS["lowess"], width=4.0, shape="spline", smoothing=1.0),
        hovertemplate="K=%{x:.2f}<br>LOWESS=%{y:.4f}<extra></extra>",
    ))

    if benchmark is not None:
        fig.add_hline(
            y=float(benchmark),
            line_dash="dot",
            line_color="rgba(96, 104, 120, 0.9)",
            annotation_text=f"benchmark={benchmark:.3f}",
            annotation_position="top left",
        )

    yaxis = dict(
        title=label,
        showline=True,
        linewidth=1,
        linecolor=PLOT_COLORS["axis"],
        gridcolor=PLOT_COLORS["grid"],
        zeroline=False,
    )
    y_range = adaptive_yaxis_range(summary, fit, metric)
    if y_range is not None:
        yaxis["range"] = y_range

    fig.update_layout(
        template="none",
        title=f"{MODEL_LABEL}: {label} vs K - 95% CI for mean with LOWESS fit",
        xaxis=dict(title="K retained per BP/RP arm", showline=True, linewidth=1, linecolor=PLOT_COLORS["axis"], gridcolor=PLOT_COLORS["grid"], zeroline=False),
        yaxis=yaxis,
        height=560,
        width=1400,
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(color=PLOT_COLORS["text"], size=14),
        legend=dict(
            orientation="h",
            x=0.01,
            y=0.99,
            bgcolor="rgba(255,255,255,0.68)",
            bordercolor="rgba(47,52,65,0.12)",
            borderwidth=1,
        ),
        margin=dict(l=80, r=45, t=75, b=65),
    )
    fig.show()

    output_stem = FIGURE_DIR / f"{safe_filename(MODEL_LABEL.lower())}_{safe_filename(metric)}_vs_K"
    html_path = output_stem.with_suffix(".html")
    fig.write_html(html_path, include_plotlyjs="cdn")
    print("Saved HTML:", html_path)

    if save_png:
        png_path = output_stem.with_suffix(".png")
        try:
            fig.write_image(png_path, width=1400, height=560, scale=2)
            print("Saved PNG:", png_path)
        except Exception as exc:
            message = str(exc).lower()
            if "kaleido" in message:
                print("PNG export skipped: install or update kaleido to enable static image export.")
            elif "chrome" in message or "chromium" in message or "plotly_get_chrome" in message:
                print("PNG export skipped: Kaleido needs Chrome/Chromium. Run `plotly_get_chrome` in this environment.")
            else:
                raise

    return fig, summary, fit


plot_outputs = {}
for metric in METRICS_TO_PLOT:
    fig, metric_summary, metric_fit = plot_metric_vs_k(res, metric, frac=0.22, save_png=True)
    if fig is not None:
        plot_outputs[metric] = {"figure": fig, "summary": metric_summary, "lowess": metric_fit}

print("Plotted metrics:", list(plot_outputs))
